In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd() / 'my-project'

files = {
    'requirements.txt': '''
fastapi
uvicorn[standard]
streamlit
requests
transformers
torch
sentencepiece
''',
    'app/__init__.py': '''
# FastAPI app package
''',
    'app/auth.py': '''
from fastapi import Header, HTTPException

VALID_API_KEYS = {
    'test-key-001': '사용자A',
    'test-key-002': '사용자B',
}


async def verify_api_key(
    x_api_key: str | None = Header(
        default=None,
        alias='X-API-Key',
    ),
) -> str:
    if x_api_key is None:
        raise HTTPException(
            status_code=401,
            detail=(
                'API Key가 필요합니다. '
                'X-API-Key 헤더를 포함해 주세요.'
            ),
        )

    if x_api_key not in VALID_API_KEYS:
        raise HTTPException(
            status_code=401,
            detail='유효하지 않은 API Key입니다.',
        )

    return VALID_API_KEYS[x_api_key]
''',
    'app/schemas.py': '''
from pydantic import BaseModel, Field


class SummarizeRequest(BaseModel):
    text: str = Field(
        min_length=10,
        max_length=10_000,
        description='요약할 한국어 원문',
    )
    max_new_tokens: int = Field(
        default=64,
        ge=10,
        le=128,
        description='요약 최대 길이',
    )


class SummarizeResponse(BaseModel):
    user: str
    summary: str
    input_char_count: int
''',
    'app/model_service.py': '''
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

MODEL_ID = 'eenzeenee/t5-small-korean-summarization'
MAX_INPUT_TOKENS = 512


class ModelLoadError(RuntimeError):
    pass


class ModelNotReadyError(RuntimeError):
    pass


class TextTooLongError(ValueError):
    pass


class KoreanSummarizer:
    def __init__(self):
        self.tokenizer = None
        self.model = None
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'

    @property
    def ready(self) -> bool:
        return self.tokenizer is not None and self.model is not None

    def load(self) -> None:
        try:
            self.tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
            self.model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)
            self.model.to(self.device)
            self.model.eval()
        except Exception as exc:
            self.tokenizer = None
            self.model = None
            raise ModelLoadError('요약 모델을 불러오지 못했습니다.') from exc

    def summarize(self, text: str, max_new_tokens: int) -> str:
        if not self.ready:
            raise ModelNotReadyError('모델이 아직 준비되지 않았습니다.')

        source = f'summarize: {text.strip()}'
        token_ids = self.tokenizer(
            source,
            truncation=False,
            add_special_tokens=True,
        )['input_ids']

        if len(token_ids) > MAX_INPUT_TOKENS:
            raise TextTooLongError(
                f'입력이 너무 깁니다. 최대 {MAX_INPUT_TOKENS} 토큰까지 가능합니다.'
            )

        inputs = self.tokenizer(
            source,
            max_length=MAX_INPUT_TOKENS,
            truncation=True,
            return_tensors='pt',
        )
        inputs = {key: value.to(self.device) for key, value in inputs.items()}

        with torch.inference_mode():
            output = self.model.generate(
                **inputs,
                num_beams=4,
                max_new_tokens=max_new_tokens,
                no_repeat_ngram_size=3,
                early_stopping=True,
            )

        return self.tokenizer.decode(
            output[0].detach().cpu(),
            skip_special_tokens=True,
        ).strip()


model_service = KoreanSummarizer()
''',
    'app/main.py': '''
import asyncio
import logging
from contextlib import asynccontextmanager

from fastapi import Depends, FastAPI, HTTPException
from starlette.concurrency import run_in_threadpool

from app.auth import verify_api_key
from app.model_service import (
    ModelLoadError,
    ModelNotReadyError,
    TextTooLongError,
    model_service,
)
from app.schemas import SummarizeRequest, SummarizeResponse

logger = logging.getLogger(__name__)
inference_slots = asyncio.Semaphore(1)


@asynccontextmanager
async def lifespan(app: FastAPI):
    try:
        model_service.load()
        logger.info('요약 모델 로드 완료')
    except ModelLoadError:
        logger.exception('모델 로드 실패')

    yield


app = FastAPI(
    title='Korean Summarization API',
    version='1.0.0',
    lifespan=lifespan,
)


@app.get('/health')
def health():
    return {
        'status': 'ok' if model_service.ready else 'degraded',
        'model_ready': model_service.ready,
        'device': model_service.device,
    }


@app.post('/summarize', response_model=SummarizeResponse)
async def summarize(
    payload: SummarizeRequest,
    current_user: str = Depends(verify_api_key),
):
    if not model_service.ready:
        raise HTTPException(
            status_code=503,
            detail='요약 모델이 준비되지 않았습니다.',
        )

    acquired = False

    try:
        await asyncio.wait_for(inference_slots.acquire(), timeout=0.1)
        acquired = True
    except TimeoutError:
        raise HTTPException(
            status_code=429,
            detail='다른 요약 요청을 처리 중입니다. 잠시 후 다시 시도해 주세요.',
        )

    try:
        summary = await run_in_threadpool(
            model_service.summarize,
            payload.text,
            payload.max_new_tokens,
        )

        if not summary:
            raise HTTPException(
                status_code=500,
                detail='요약 결과가 비어 있습니다.',
            )

        return SummarizeResponse(
            user=current_user,
            summary=summary,
            input_char_count=len(payload.text),
        )

    except TextTooLongError as exc:
        raise HTTPException(status_code=422, detail=str(exc))
    except ModelNotReadyError as exc:
        raise HTTPException(status_code=503, detail=str(exc))
    except HTTPException:
        raise
    except Exception:
        logger.exception('요약 처리 중 서버 오류')
        raise HTTPException(
            status_code=500,
            detail='요약 처리 중 서버 오류가 발생했습니다.',
        )
    finally:
        if acquired:
            inference_slots.release()
''',
    'frontend/app.py': '''
import requests
import streamlit as st

API_URL = 'http://127.0.0.1:8000/summarize'

st.set_page_config(page_title='한국어 요약 서비스', page_icon='📝')
st.title('📝 한국어 텍스트 요약')
st.caption('FastAPI + Hugging Face T5 + API Key 인증')

with st.form('summarize_form'):
    api_key = st.text_input(
        'API Key',
        type='password',
        placeholder='예: test-key-001',
    )

    text = st.text_area(
        '요약할 텍스트',
        height=240,
        placeholder='10자 이상의 한국어 원문을 입력하세요.',
    )

    max_new_tokens = st.slider(
        '요약 최대 길이',
        min_value=10,
        max_value=128,
        value=64,
    )

    submitted = st.form_submit_button('요약하기', type='primary')

if submitted:
    if not text.strip():
        st.warning('요약할 텍스트를 입력해 주세요.')
        st.stop()

    headers = {'X-API-Key': api_key} if api_key else {}

    try:
        with st.spinner('요약 중입니다...'):
            response = requests.post(
                API_URL,
                headers=headers,
                json={
                    'text': text,
                    'max_new_tokens': max_new_tokens,
                },
                timeout=(3, 90),
            )

        if response.ok:
            data = response.json()
            st.success(f'{data["user"]}님, 요약이 완료되었습니다.')
            st.subheader('요약 결과')
            st.write(data['summary'])
            st.caption(f'입력 글자 수: {data["input_char_count"]}')

        else:
            try:
                detail = response.json().get('detail', '알 수 없는 오류')
            except ValueError:
                detail = '서버 응답을 해석할 수 없습니다.'

            messages = {
                401: 'API Key가 없거나 유효하지 않습니다.',
                422: '입력값이 올바르지 않거나 텍스트가 너무 깁니다.',
                429: '요청이 많습니다. 잠시 후 다시 시도해 주세요.',
                500: '서버 내부 오류가 발생했습니다.',
                503: '현재 모델을 사용할 수 없습니다.',
            }

            st.error(messages.get(response.status_code, detail))
            with st.expander('오류 상세'):
                st.code(f'HTTP {response.status_code}\n{detail}')

    except requests.exceptions.ConnectionError:
        st.error('FastAPI 서버에 연결할 수 없습니다. 서버 실행 상태를 확인해 주세요.')
    except requests.exceptions.Timeout:
        st.error('요청 시간이 초과되었습니다. 텍스트를 줄여 다시 시도해 주세요.')
    except requests.RequestException as exc:
        st.error('네트워크 요청 중 오류가 발생했습니다.')
        with st.expander('오류 상세'):
            st.code(str(exc))
''',
}

for relative_path, content in files.items():
    path = PROJECT_ROOT / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content.strip() + '\n', encoding='utf-8')

print(f'프로젝트 생성 완료: {PROJECT_ROOT}')


프로젝트 생성 완료: /content/my-project
